In [ ]:
import os
import random
import scipy.io as sio
import numpy as np
import cv2
import pywt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.signal import butter, filtfilt, resample
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import VGG16_Weights
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def butter_highpass_filter(data, cutoff=10.0, fs=25600.0, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, data)
    return y

In [ ]:

class TargetDataset(Dataset):
    def __init__(self, data_list, labels, original_fs, target_fs, segment_length=1200, step_size=None, transform=None):
        self.samples = []
        self.transform = transform
        self.target_fs = target_fs
        
        if step_size is None:
            step_size = segment_length
            
        for signal, label in zip(data_list, labels):
            filtered_signal = butter_highpass_filter(signal, cutoff=10.0, fs=original_fs)
            
            if target_fs < original_fs:
                num_samples = int(len(filtered_signal) * (target_fs / original_fs))
                resampled_signal = resample(filtered_signal, num_samples)
            else:
                resampled_signal = filtered_signal
                
            if len(resampled_signal) >= segment_length:
                num_segments = (len(resampled_signal) - segment_length) // step_size + 1
                for i in range(num_segments):
                    start_idx = i * step_size
                    segment = resampled_signal[start_idx : start_idx + segment_length]
                    self.samples.append((segment, label))

    def __len__(self):
        return len(self.samples)
    
    def generate_cwt(self, signal):
        fs = self.target_fs
        fc = 1.0
        frequencies = np.geomspace(fs/2, 10, 128)
        scales = (fc * fs) / frequencies
        wavelet = 'cmor5.0-1.0'
    
        coefficients, _ = pywt.cwt(signal, scales, wavelet, sampling_period=1/fs)
        amplitude = np.power(np.abs(coefficients), 2)
    
        amp_min, amp_max = amplitude.min(), amplitude.max()
        if amp_max > amp_min:
            normalized_amp = (amplitude - amp_min) / (amp_max - amp_min)
        else:
            normalized_amp = np.zeros_like(amplitude)
    
        img_8bit = np.uint8(normalized_amp * 255)
        img_color = cv2.applyColorMap(img_8bit, cv2.COLORMAP_JET)
        img_resized = cv2.resize(img_color, (224, 224))
        return cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    def __getitem__(self, idx):
        signal_segment, label = self.samples[idx]
        image_np = self.generate_cwt(signal_segment)
        
        if self.transform:
            image_tensor = self.transform(image_np)
        else:
            transform_default = transforms.Compose([transforms.ToTensor()])
            image_tensor = transform_default(image_np)
            
        return image_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
def load_xjtu_data(data_dir, num_minutes=35):
    data_list, labels = [], []
    
    b1_path = os.path.join(data_dir, 'bearing1.mat') 
    b6_path = os.path.join(data_dir, 'bearing6.mat') 
    b4_path = os.path.join(data_dir, 'bearing4.mat') 
    
    b1_mat = sio.loadmat(b1_path)['rawnet']
    b6_mat = sio.loadmat(b6_path)['rawnet']
    b4_mat = sio.loadmat(b4_path)['rawnet']
    
    total_mins_b1 = b1_mat.shape[2]
    assert total_mins_b1 >= num_minutes * 2, f"Label contradiction: B1 requires at least {num_minutes * 2} mins!"
    
    for i in range(num_minutes):
        data_list.append(b1_mat[:, 0, i].flatten())
        labels.append(0)
        
    total_mins = b6_mat.shape[2]
    for i in range(total_mins - num_minutes, total_mins):
        data_list.append(b6_mat[:, 0, i].flatten())
        labels.append(1)
        
    for i in range(total_mins_b1 - num_minutes, total_mins_b1):
        data_list.append(b1_mat[:, 0, i].flatten())
        labels.append(2)

    total_mins = b4_mat.shape[2]
    for i in range(total_mins - num_minutes, total_mins):
        data_list.append(b4_mat[:, 0, i].flatten())
        labels.append(3)
        
    return data_list, labels

xjtu_dir = "/kaggle/input/datasets/onkarraskar/xjtu-data/XJTU_orig/originaldata"
print("Parsing XJTU-SY data files...")
xjtu_signals, xjtu_labels = load_xjtu_data(xjtu_dir, num_minutes=35)

signals_by_class = {0: [], 1: [], 2: [], 3: []}
for sig, lbl in zip(xjtu_signals, xjtu_labels):
    signals_by_class[lbl].append(sig)

train_sigs, train_lbls = [], []
val_sigs, val_lbls = [], []
test_sigs, test_lbls = [], []

for lbl, sigs in signals_by_class.items():
    train_sigs.extend(sigs[:8]);    train_lbls.extend([lbl]*8)
    val_sigs.extend(sigs[8:14]);    val_lbls.extend([lbl]*6)
    test_sigs.extend(sigs[14:35]);  test_lbls.extend([lbl]*21)

print("Initializing Datasets...")
train_dataset = TargetDataset(train_sigs, train_lbls, original_fs=25600.0, target_fs=4334.38, segment_length=600, step_size=600)
val_dataset   = TargetDataset(val_sigs, val_lbls, original_fs=25600.0, target_fs=4334.38, segment_length=600, step_size=600)
test_dataset  = TargetDataset(test_sigs, test_lbls, original_fs=25600.0, target_fs=4334.38, segment_length=600, step_size=600)


batch_size = 32
xjtu_train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
xjtu_val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
xjtu_test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


print(f"XJTU-SY Data Split summary :")
print(f"  --> Training Samples:   {len(train_dataset)}")
print(f"  --> Validation Samples: {len(val_dataset)}")
print(f"  --> Testing Samples:    {len(test_dataset)}")



In [ ]:
transfer_model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
in_features = transfer_model.classifier[6].in_features

transfer_model.classifier[6] = nn.Linear(in_features, 10)
transfer_model.load_state_dict(torch.load('/kaggle/input/datasets/onkarraskar/cwru-best-weight/cwru_pretrained_vgg16 (1).pth', map_location=device))
print("Successfully loaded pre-trained CWRU baseline weights.")

transfer_model.classifier[6] = nn.Linear(in_features, 4)
transfer_model = transfer_model.to(device)

for param in transfer_model.features.parameters():
    param.requires_grad = False
for param in transfer_model.classifier.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_stage1 = optim.Adam(transfer_model.classifier.parameters(), lr=1e-5)

num_epochs_stage1 = 20
best_val_loss = float('inf')
patience = 5
epochs_no_improve = 0

print("\nStarting Stage 1: Training Fully Connected Layers only...")
for epoch in range(num_epochs_stage1):
    transfer_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in xjtu_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_stage1.zero_grad()
        
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stage1.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100. * correct / total
    
    transfer_model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in xjtu_val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = transfer_model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / total_val
    val_acc = 100. * correct_val / total_val
    
    print(f"Epoch [{epoch+1:02d}/{num_epochs_stage1:02d}] | Train Loss: {running_loss/total:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(transfer_model.state_dict(), 'xjtu_stage1_vgg16.pth')
        print("   --> Saved best Stage 1 model.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"   --> [!] Early stopping triggered at epoch {epoch+1}.")
            break

print("\nStage 1 Complete.")

In [ ]:
transfer_model.load_state_dict(torch.load('xjtu_stage1_vgg16.pth', map_location=device))

for param in transfer_model.features[24:].parameters():
    param.requires_grad = True

optimizer_stage2 = optim.Adam(filter(lambda p: p.requires_grad, transfer_model.parameters()), lr=1e-5)
best_val_loss = float('inf')
epochs_no_improve = 0
patience = 4

print("\nStarting Stage 2: Fine-Tuning Conv5 and Classifier...")
for epoch in range(15):
    transfer_model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in xjtu_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_stage2.zero_grad()
        
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_stage2.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100. * correct / total
    
    transfer_model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in xjtu_val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = transfer_model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / total_val
    val_acc = 100. * correct_val / total_val
    
    print(f"Epoch [{epoch+1:02d}/15] | Train Loss: {running_loss/total:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} - Val Acc: {val_acc:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(transfer_model.state_dict(), 'xjtu_stage2_vgg16.pth')
        print("   --> Saved best Stage 2 model.")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("   --> [!] Early stopping triggered.")
            break

In [ ]:
print("========STAGE 2 COMPLETE. INITIATING TESTING=========")

transfer_model.load_state_dict(torch.load('xjtu_stage2_vgg16.pth', map_location=device))
transfer_model.eval()

all_preds, all_labels = [], []
test_correct, test_loss, total_test = 0, 0.0, 0

with torch.no_grad():
    for inputs, labels in xjtu_test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = transfer_model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        
        total_test += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

avg_test_loss = test_loss / total_test
final_test_acc = 100. * test_correct / total_test

print(f"TEST LOSS: {avg_test_loss:.4f}")
print(f"TEST ACCURACY (XJTU-SY Dataset): {final_test_acc:.2f}% \n")

cm = confusion_matrix(all_labels, all_preds)
class_names = ['Healthy', 'Inner Race', 'Outer Race', 'Cage']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'XJTU-SY Test Confusion Matrix (Acc: {final_test_acc:.2f}%)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))